In [ ]:
import os
from google.colab import userdata

os.environ["GH_USER"] = userdata.get("GH_USER")
os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")

In [ ]:
import os, textwrap

askpass_path = "/tmp/gh_askpass.sh"
with open(askpass_path, "w") as f:
    f.write(textwrap.dedent("""\
        #!/bin/sh
        case "$1" in
          *Username*) echo "$GH_USER" ;;
          *Password*) echo "$GH_TOKEN" ;;
          *) echo "" ;;
        esac
    """))
os.chmod(askpass_path, 0o700)
os.environ["GIT_ASKPASS"] = askpass_path
os.environ["GIT_TERMINAL_PROMPT"] = "0"

!git clone https://github.com/incredet/HyperSpectralSuperResolution.git
%cd /content/HyperSpectralSuperResolution/

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import json
import shutil
from pathlib import Path
from pprint import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from joblib import Parallel, delayed
from tqdm import tqdm

from documentation.config import PipelineConfig
from documentation.pairs_artifacts import load_pipeline_config
from documentation.report_builder import R2Aggregator, plot_r2_histogram, plot_r2_per_band
from data.EMIT.emit_utils import closest_bands
from spectral.s2_to_emit import fit_tile, plot_spectral_match

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
DRIVE_ROOT = Path("/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches")
RUN_TAG    = "2026-04-02"
MODE       = "downsample"

In [ ]:
drive_base = DRIVE_ROOT / RUN_TAG
config_dict = load_pipeline_config(drive_base)
config = PipelineConfig.from_dict({
    **config_dict,
    "aoi_lat": 0.0, "aoi_lon": 0.0,        # dummy — not used here
    "drive_root": str(drive_base),
})

spectral_params = config.spectral_params

print("Spectral params from pipeline_config.yaml:")
pprint(spectral_params)
print(f"Mode: {MODE}")

In [ ]:
manifests = sorted(drive_base.glob("*/*/manifest.csv"))
print(f"Found {len(manifests)} pair manifest(s) under {drive_base}")

In [ ]:
LOCAL_STAGE = Path("/content/_cm_stage")
N_JOBS = 6

all_r2_rows = []

def _process_tile(item, _spectral_params=spectral_params, _mode=MODE):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    try:
        matcher, stats = fit_tile(
            s2_tile_path=item["s2_local"],
            emit_b32_tile_path=item["b32_local"],
            **_spectral_params,
            mode=_mode,
            verbose=False,
        )
    except ValueError as e:
        return {"tile_idx": item["tile_idx"], "status": "skipped", "error": str(e)}

    matcher.save(item["matcher_local"])
    matcher.apply_to_tile(item["s2_local"], out_path=item["out_local"])

    plot_spectral_match(
        item["s2_local"], item["out_local"], item["b32_local"],
        title_suffix=f"tile {item['tile_idx']:03d}",
        r2_mean=stats["r2_mean"],
        save_path=item["plot_local"],
        show=False,
        wavelengths_nm=item["wl_b32"],
    )
    plt.close("all")

    return {
        "tile_idx":    item["tile_idx"],
        "status":      "ok",
        "r2_mean":     stats["r2_mean"],
        "r2_per_band": stats["r2_per_band"],
    }

In [ ]:
for manifest_path in tqdm(manifests, desc="Pairs"):
    pair_dir = manifest_path.parent
    pair_id  = pair_dir.name
    aoi_slug_name = pair_dir.parent.name
    pair_key = f"{aoi_slug_name}/{pair_id}"

    mf = pd.read_csv(manifest_path)
    if "emit_b32_tif" not in mf.columns or "s2_tif" not in mf.columns:
        print(f"  {pair_key}: manifest missing required columns — skipping")
        continue

    synth_drive   = pair_dir / "regression_synth"
    matcher_drive = pair_dir / "regression_matchers"
    plots_drive   = pair_dir / "plots"
    synth_drive.mkdir(parents=True, exist_ok=True)
    matcher_drive.mkdir(parents=True, exist_ok=True)
    plots_drive.mkdir(parents=True, exist_ok=True)

    wl_path = list(pair_dir.glob("metadata/*.wavelengths.json")) or \
              list(pair_dir.glob("**/*.wavelengths.json"))
    if wl_path:
        with open(wl_path[0]) as f:
            wl_nm = np.array(json.load(f))
    else:
        first_emit = mf["emit_b32_tif"].dropna().iloc[0]
        with rasterio.open(first_emit) as src:
            wl_nm = np.array([float(d.split(" ")[0]) for d in src.descriptions if d])
        if len(wl_nm) == 0:
            print(f"  {pair_key}: cannot determine wavelengths — skipping")
            continue

    first_b32 = mf["emit_b32_tif"].dropna().iloc[0]
    with rasterio.open(first_b32) as src:
        n_bands_b32 = src.count
    if len(wl_nm) > n_bands_b32:
        meta_jsons = sorted(pair_dir.glob("metadata/tiles/*.json"))
        if meta_jsons:
            with open(meta_jsons[0]) as f:
                tile_meta = json.load(f)
            b32_idx = tile_meta.get("emit_b32_indices_0based")
            if b32_idx:
                wl_b32 = wl_nm[b32_idx]
            else:
                idxs_0, _ = closest_bands(wl_nm, list(config.emit_target_wavelengths_nm))
                b32_idx = sorted(set(idxs_0))
                wl_b32 = wl_nm[b32_idx]
        else:
            idxs_0, _ = closest_bands(wl_nm, list(config.emit_target_wavelengths_nm))
            b32_idx = sorted(set(idxs_0))
            wl_b32 = wl_nm[b32_idx]
    else:
        wl_b32 = wl_nm

    pair_local = LOCAL_STAGE / pair_id
    if pair_local.exists():
        shutil.rmtree(pair_local)
    pair_local.mkdir(parents=True)
    synth_local   = pair_local / "regression_synth";   synth_local.mkdir()
    matcher_local = pair_local / "regression_matchers"; matcher_local.mkdir()
    plots_local   = pair_local / "plots";               plots_local.mkdir()

    work = []
    for _, row in mf.iterrows():
        tile_idx = int(row["idx"])
        s2_tif   = row["s2_tif"]
        b32_tif  = row.get("emit_b32_tif")
        if pd.isna(b32_tif) or pd.isna(s2_tif):
            continue
        if not Path(b32_tif).exists() or not Path(s2_tif).exists():
            continue

        s2_local  = pair_local / Path(s2_tif).name
        b32_local = pair_local / Path(b32_tif).name
        if not s2_local.exists():
            shutil.copy2(s2_tif, s2_local)
        if not b32_local.exists():
            shutil.copy2(b32_tif, b32_local)

        out_name = Path(s2_tif).name.replace("_s2.tif", "_regression_synth.tif")
        work.append({
            "tile_idx":      tile_idx,
            "s2_local":      str(s2_local),
            "b32_local":     str(b32_local),
            "out_local":     str(synth_local / out_name),
            "out_drive":     str(synth_drive / out_name),
            "matcher_local": str(matcher_local / f"tile_{tile_idx:03d}_matcher.joblib"),
            "matcher_drive": str(matcher_drive / f"tile_{tile_idx:03d}_matcher.joblib"),
            "plot_local":    str(plots_local / f"{pair_id}_tile{tile_idx:03d}_synth.png"),
            "plot_drive":    str(plots_drive / f"{pair_id}_tile{tile_idx:03d}_synth.png"),
            "wl_b32":        wl_b32,
        })

    if not work:
        shutil.rmtree(pair_local, ignore_errors=True)
        print(f"  {pair_key}: no valid tiles")
        continue

    results_tiles = Parallel(n_jobs=N_JOBS, prefer="processes")(
        delayed(_process_tile)(item) for item in work
    )

    r2_agg = R2Aggregator(wavelengths_nm=wl_b32)
    n_fitted = 0
    for item, res in zip(work, results_tiles):
        if res["status"] == "skipped":
            print(f"    tile {res['tile_idx']:03d}: SKIPPED — {res['error']}")
            continue
        n_fitted += 1
        r2_agg.add(res["tile_idx"], np.array(res["r2_per_band"]), res["r2_mean"])
        all_r2_rows.append({
            "aoi_slug": aoi_slug_name, "pair_id": pair_id,
            "tile_idx": res["tile_idx"], "r2_mean": res["r2_mean"],
            **{f"r2_band_{i:02d}": v for i, v in enumerate(res["r2_per_band"])},
        })
        for src_key, dst_key in [("out_local", "out_drive"),
                                  ("matcher_local", "matcher_drive"),
                                  ("plot_local", "plot_drive")]:
            src = Path(item[src_key])
            if src.exists():
                shutil.copy2(src, item[dst_key])

    if n_fitted > 0:
        r2_summary = r2_agg.summary()
        print(f"  {pair_key}: {n_fitted} tiles, "
              f"R² mean={r2_summary['global_mean']:.4f}, "
              f"median={r2_summary['global_median']:.4f}")
        plot_r2_histogram(r2_agg.r2_mean, plots_drive / "r2_histogram.png",
                          title=f"R² distribution — {pair_id}")
        plot_r2_per_band(r2_agg.r2_per_band, r2_agg.wavelengths_nm,
                         plots_drive / "r2_per_band.png")
        plt.close("all")
    else:
        print(f"  {pair_key}: no tiles fitted")

    # Free local disk for next pair
    shutil.rmtree(pair_local, ignore_errors=True)

shutil.rmtree(LOCAL_STAGE, ignore_errors=True)
print(f"\nDone. {len(all_r2_rows)} tiles processed across {len(manifests)} pairs.")

In [ ]:
r2_df = pd.DataFrame(all_r2_rows)

r2_csv_path = drive_base / "r2_all_tiles.csv"
r2_df.to_csv(r2_csv_path, index=False)

print(f"R² CSV saved: {r2_csv_path}  ({len(r2_df)} tiles)")
print(f"\nR² mean summary:")
print(f"  Mean:   {r2_df['r2_mean'].mean():.4f}")
print(f"  Median: {r2_df['r2_mean'].median():.4f}")
print(f"  Min:    {r2_df['r2_mean'].min():.4f}")
print(f"  Max:    {r2_df['r2_mean'].max():.4f}")
print(f"  Std:    {r2_df['r2_mean'].std():.4f}")

r2_df.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(r2_df["r2_mean"], bins=30, edgecolor="black", alpha=0.7)
axes[0].axvline(r2_df["r2_mean"].median(), color="red", ls="--",
                label=f"median={r2_df['r2_mean'].median():.3f}")
axes[0].set_xlabel("Mean R²")
axes[0].set_ylabel("Count")
axes[0].set_title("R² distribution (all tiles)")
axes[0].legend()

aoi_order = r2_df.groupby("aoi_slug")["r2_mean"].median().sort_values(ascending=False).index
r2_df_sorted = r2_df.set_index("aoi_slug").loc[aoi_order].reset_index()
r2_df_sorted.boxplot(column="r2_mean", by="aoi_slug", ax=axes[1], rot=90)
axes[1].set_title("R² by AOI")
axes[1].set_xlabel("")
axes[1].set_ylabel("Mean R²")
plt.suptitle("")

plt.tight_layout()
fig.savefig(str(drive_base / "r2_global_summary.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"\nTiles with R² >= 0.75: {(r2_df['r2_mean'] >= 0.75).sum()} / {len(r2_df)}")
print(f"Tiles with R² >= 0.80: {(r2_df['r2_mean'] >= 0.80).sum()} / {len(r2_df)}")

In [ ]:
print(f"Tiles with R² >= 0.85: {(r2_df['r2_mean'] >= 0.85).sum()} / {len(r2_df)}")